# Korean-English Translation: Training from Scratch

This notebook trains a Transformer model on the opus100 Korean-English dataset for translation task.

### Architecture
Original Transformer (Vaswani et al., 2017)
- Encoder-Decoder structure
- d_model    : 512
- num_heads  : 8
- d_ff       : 2048
- num_layers : 6
- dropout    : 0.1

### Why not Pre-training?
In practice, training a Transformer for translation does not require pre-training. The original paper trained directly on translation data from scratch — which is exactly what we do here.

Pre-training (BERT, GPT, T5) is useful when:
- Task-specific labeled data is scarce
- You want to transfer knowledge across tasks

For translation, large parallel corpora (like opus100) are available, making pre-training unnecessary.

### What this notebook does
1. Load SentencePiece tokenizers (Korean & English)
2. Load opus100 Korean-English dataset (train/val/test)
3. Train Transformer from scratch on translation task
4. Save best model checkpoint based on validation loss
5. Test translation with greedy decoding

### Dataset
- opus100 Korean-English (HuggingFace)
- Train : 1,000,000 sentence pairs
- Val   : 2,000 sentence pairs
- Test  : 2,000 sentence pairs

### Training
- Loss      : CrossEntropyLoss (label smoothing=0.1)
- Optimizer : Adam (beta1=0.9, beta2=0.98, eps=1e-9)
- LR Schedule: Warmup (4000 steps) + decay
- Batch size : 64
- Max seq len: 128

### Expected Results
- Perfect translation is not expected given compute constraints
- Simple sentences should show reasonable translation quality
- Training/validation loss should decrease steadily

### Note on Tokenizer
Separate Korean and English tokenizers are used since we train directly on translation data without pre-training. If pre-training on both languages were required, a shared tokenizer would be needed. See README for details.

## 1. Imports

In [1]:
import torch
import torch.nn as nn
import sys
import os
sys.path.append('./src')

from transformer import Transformer
from tokenizer import Tokenizer
from dataset import get_dataloader
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Load Tokenizers

In [3]:
ko_tokenizer = Tokenizer()
ko_tokenizer.load('./data/processed/ko_tokenizer.model')

en_tokenizer = Tokenizer()
en_tokenizer.load('./data/processed/en_tokenizer.model')

print(f"Korean vocab size: {ko_tokenizer.vocab_size()}")
print(f"English vocab size: {en_tokenizer.vocab_size()}")

Tokenizer loaded. Vocab size: 16000
Tokenizer loaded. Vocab size: 16000
Korean vocab size: 16000
English vocab size: 16000


## 3. Load Dataset

In [4]:
dataset = load_dataset("opus100", "en-ko")

train_loader = get_dataloader(
    dataset['train'],
    src_tokenizer=ko_tokenizer,
    tgt_tokenizer=en_tokenizer,
    batch_size=64,
    max_seq_len=128,
    shuffle=True
)

val_loader = get_dataloader(
    dataset['validation'],
    src_tokenizer=ko_tokenizer,
    tgt_tokenizer=en_tokenizer,
    batch_size=64,
    max_seq_len=128,
    shuffle=False
)

test_loader = get_dataloader(
    dataset['test'],
    src_tokenizer=ko_tokenizer,
    tgt_tokenizer=en_tokenizer,
    batch_size=64,
    max_seq_len=128,
    shuffle=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

Train batches: 15625
Val batches  : 32
Test batches : 32


## 4. Model

In [5]:
# Training from scratch — no pretrained checkpoint
model = Transformer(
    src_vocab_size=ko_tokenizer.vocab_size(),
    tgt_vocab_size=en_tokenizer.vocab_size(),
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6,
    dropout=0.1,
    max_seq_len=128
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 68,730,496


## 5. Loss & Optimizer

In [13]:
criterion = nn.CrossEntropyLoss(
    ignore_index=en_tokenizer.PAD_IDX,
    label_smoothing=0.1
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1.0,
    betas=(0.9, 0.98),
    eps=1e-9
)

# Learning rate warmup schedule (from paper)
def get_lr(step, d_model=512, warmup_steps=4000):
    if step == 0:
        step = 1
    return d_model**(-0.5) * min(step**(-0.5), step * warmup_steps**(-1.5))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: get_lr(step)
)

## 6. Training & Validation Functions

In [14]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0

    for batch_idx, (src, tgt, src_mask, tgt_mask) in enumerate(loader):
        src      = src.to(device)
        tgt      = tgt.to(device)
        src_mask = src_mask.to(device)
        tgt_mask = tgt_mask.to(device)

        # Teacher forcing
        # tgt_in : [BOS, w1, w2, ...]
        # tgt_out: [w1, w2, ..., EOS]
        tgt_in      = tgt[:, :-1]
        tgt_out     = tgt[:, 1:]
        tgt_mask_in = tgt_mask[:, :, :, :-1]

        output  = model(src, tgt_in, src_mask, tgt_mask_in)
        output  = output.reshape(-1, output.size(-1))
        tgt_out = tgt_out.reshape(-1)

        loss = criterion(output, tgt_out)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
          
            print(f"  Batch {batch_idx}/{len(loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")
                #   f"LR: {optimizer.param_groups[0]['lr'] :.6f}")

    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt, src_mask, tgt_mask in loader:
            src      = src.to(device)
            tgt      = tgt.to(device)
            src_mask = src_mask.to(device)
            tgt_mask = tgt_mask.to(device)

            tgt_in      = tgt[:, :-1]
            tgt_out     = tgt[:, 1:]
            tgt_mask_in = tgt_mask[:, :, :, :-1]

            output  = model(src, tgt_in, src_mask, tgt_mask_in)
            output  = output.reshape(-1, output.size(-1))
            tgt_out = tgt_out.reshape(-1)

            loss = criterion(output, tgt_out)
            total_loss += loss.item()

    return total_loss / len(loader)

## 7. Run Training

In [15]:
os.makedirs('./data/checkpoints', exist_ok=True)

num_epochs    = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    train_loss = train_epoch(model, train_loader, optimizer,
                             scheduler, criterion, device)
    val_loss   = validate(model, val_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss'            : val_loss,
        }, './data/checkpoints/best_model.pt')
        print(f"  Best model saved (val_loss: {val_loss:.4f})")


Epoch 1/10
--------------------------------------------------
  Batch 0/15625 | Loss: 9.9367 | LR: 0.000000
  Batch 100/15625 | Loss: 7.8359 | LR: 0.000018
  Batch 200/15625 | Loss: 6.9194 | LR: 0.000035
  Batch 300/15625 | Loss: 6.2358 | LR: 0.000053
  Batch 400/15625 | Loss: 5.5334 | LR: 0.000070
  Batch 500/15625 | Loss: 5.7729 | LR: 0.000088
  Batch 600/15625 | Loss: 5.6279 | LR: 0.000105
  Batch 700/15625 | Loss: 5.8377 | LR: 0.000122
  Batch 800/15625 | Loss: 5.3782 | LR: 0.000140
  Batch 900/15625 | Loss: 5.6417 | LR: 0.000157
  Batch 1000/15625 | Loss: 5.3336 | LR: 0.000175
  Batch 1100/15625 | Loss: 5.4388 | LR: 0.000192
  Batch 1200/15625 | Loss: 5.3621 | LR: 0.000210
  Batch 1300/15625 | Loss: 5.5595 | LR: 0.000227
  Batch 1400/15625 | Loss: 5.2342 | LR: 0.000245
  Batch 1500/15625 | Loss: 5.2048 | LR: 0.000262
  Batch 1600/15625 | Loss: 5.3286 | LR: 0.000280
  Batch 1700/15625 | Loss: 5.2114 | LR: 0.000297
  Batch 1800/15625 | Loss: 5.3545 | LR: 0.000315
  Batch 1900/15625

## 8. Inference (Greedy Decoding)

In [ ]:
def translate(model, sentence, src_tokenizer, tgt_tokenizer,
              max_len=128, device='cuda'):
    model.eval()

    # Tokenize source sentence (Korean)
    src_ids  = src_tokenizer.encode(sentence)
    src      = torch.tensor(src_ids).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src, src_tokenizer.PAD_IDX)

    # Encode source
    with torch.no_grad():
        encoder_output = model.encode(src, src_mask)

    # Decode autoregressively
    tgt_ids = [tgt_tokenizer.BOS_IDX]

    for _ in range(max_len):
        tgt      = torch.tensor(tgt_ids).unsqueeze(0).to(device)
        tgt_mask = model.make_tgt_mask(tgt, tgt_tokenizer.PAD_IDX)

        with torch.no_grad():
            output     = model.decode(tgt, encoder_output, src_mask, tgt_mask)
            logits     = model.output_linear(output)

        # Greedy decoding
        next_token = logits[:, -1, :].argmax(dim=-1).item()
        tgt_ids.append(next_token)

        if next_token == tgt_tokenizer.EOS_IDX:
            break

    return tgt_tokenizer.decode(tgt_ids)


# Test translation
sentences = [
    "나는 학교에 갑니다.",
    "오늘 날씨가 좋습니다.",
    "저는 한국어를 공부합니다.",
]

for sentence in sentences:
    translation = translate(model, sentence, ko_tokenizer,
                            en_tokenizer, device=device)
    print(f"KO: {sentence}")
    print(f"EN: {translation}")
    print()

KO: 나는 학교에 갑니다.
EN: I'm sorry.

KO: 오늘 날씨가 좋습니다.
EN: I'm sorry.

KO: 저는 한국어를 공부합니다.
EN: I'm sorry.

